# Molecule similarity-graph message passing — results

**Compute runs externally** (GPU server); this notebook only loads the results and plots them.

A kNN graph in molecule (ChemBERTa) space lets each molecule borrow from chemically similar ones via
one message-passing step. Three molecule representations are probed with XGBoost (protein stays raw):

- **raw** — `[raw mol | raw prot]`, no graph.
- **mp_untrained** — parameter-free 1-hop propagation `z = Â·X` (graph, no training).
- **mp_trained** — 1-layer `SAGEConv` + MLP link head trained on the task, head dropped, boost on `z_mol`.

Both regimes, **5 repeats** each (transductive = LoRaX folds 1–5; inductive = cold-split seeds 42–46).
The inductive cold val/test split is **stratified to matched prevalence**. Embeddings are standardised.

### Run the compute
```bash
bash scripts/modeling/eval/run_molecule_graph_mp.sh            # GPU, live dashboard
MAX_PARALLEL=4 bash scripts/modeling/eval/run_molecule_graph_mp.sh
```
Then run the cells below.

In [ ]:
import sys, pathlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('run from inside the orbind repository')
    ROOT = ROOT.parent

RESULT_DIR = ROOT / 'results/graph/mol_graph_mp'
COMBINED = RESULT_DIR / 'tables/molecule_graph_mp_runs.csv'
RUNS_DIR = RESULT_DIR / 'tables/runs'
FIGURES_DIR = ROOT / 'notebooks/figures'; FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
REGIMES = ['transductive', 'inductive_molecule']

# prefer the combined CSV; fall back to globbing per-unit CSVs (partial runs still render)
if COMBINED.exists():
    RES = pd.read_csv(COMBINED)
    print('loaded', COMBINED.relative_to(ROOT))
elif RUNS_DIR.exists() and glob.glob(str(RUNS_DIR / '*.csv')):
    RES = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RUNS_DIR / '*.csv')))], ignore_index=True)
    print(f'loaded {RES.repeat.nunique()} repeat-files from', RUNS_DIR.relative_to(ROOT))
else:
    RES = pd.DataFrame()
    print('No results yet. Run:  bash scripts/modeling/eval/run_molecule_graph_mp.sh')

def ci95_half(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

if len(RES):
    n = RES.groupby('regime').repeat.nunique().to_dict()
    print('repeats per regime:', n)
    print('mean test prevalence:', RES.groupby('regime').test_prevalence.mean().round(3).to_dict())
    display(RES.groupby(['regime', 'bar'])[METRICS].mean().round(3))

In [ ]:
# ---- bar-grid: regimes (rows) x metrics (cols), 3 bars, mean +/- 95% t-CI over repeats ----
BARS = ['raw', 'mp_untrained', 'mp_trained']
LABEL = {'raw': 'raw\nmol', 'mp_untrained': 'MP\n(untrained)', 'mp_trained': 'MP\n(trained)'}
BAR_C = {'raw': '#9a9a95', 'mp_untrained': '#86b6ef', 'mp_trained': '#2a78d6'}

if not len(RES):
    print('No results to plot yet.')
else:
    fig, axes = plt.subplots(len(REGIMES), len(METRICS), figsize=(14, 7), squeeze=False)
    for ri, regime in enumerate(REGIMES):
        d = RES[RES.regime == regime]
        for mi, metric in enumerate(METRICS):
            ax = axes[ri, mi]
            means = [d[d.bar == b][metric].mean() for b in BARS]
            errs = [ci95_half(d[d.bar == b][metric]) for b in BARS]
            x = np.arange(len(BARS))
            ax.bar(x, means, color=[BAR_C[b] for b in BARS], edgecolor='white', linewidth=1.2,
                   yerr=np.nan_to_num(errs), error_kw=dict(ecolor='#33322f', capsize=3, lw=1), zorder=3)
            ax.axhline(means[0], color=BAR_C['raw'], ls='--', lw=1, zorder=1)
            ax.set_ylim(0, 1); ax.grid(alpha=.15, axis='y')
            ax.set_xticks(x); ax.set_xticklabels([LABEL[b] for b in BARS], fontsize=8)
            for s in ('top', 'right'): ax.spines[s].set_visible(False)
            if ri == 0: ax.set_title(metric, fontsize=11)
            for xi, v in zip(x, means):
                if np.isfinite(v):
                    ax.text(xi, v + .02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        nrep = d.repeat.nunique()
        axes[ri, 0].set_ylabel(f'{regime}\n({nrep} repeats)', fontsize=10)
    fig.suptitle('Molecule similarity-graph MP — mean +/- 95% t-CI (stratified inductive split; '
                 'dashed = raw baseline)', fontsize=12.5, y=1.0)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'molecule_similarity_graph_mp.png', dpi=150, bbox_inches='tight')
    plt.show()

## How to read

- **transductive** — expected flat vs raw (all molecules seen; boost already reads molecule identity).
- **inductive_molecule** — the real test. `mp_untrained > raw` = the chemistry neighbourhood gives cold
  molecules free transferable signal; `mp_trained > mp_untrained` = the learned pass adds beyond it.
- Compare on **AUROC** (prevalence-independent) primarily; the stratified split keeps val/test
  prevalence matched so AUPRC is comparable across repeats too. Whiskers are 95% t-CI over 5 repeats.

Re-run the compute (knobs via env): `K`, `EPOCHS`, `HIDDEN`, `MAX_PARALLEL`, `DEVICE`. Delete
`results/graph/mol_graph_mp/tables/` to regenerate.